# 1. Konfigurasi & Load Environment Variables

In [1]:
import os 
from dotenv import load_dotenv
from pathlib import Path
import glob
from sentence_transformers import SentenceTransformer
import numpy as np
import re
from tqdm.auto import tqdm
from qdrant_client import QdrantClient
from qdrant_client.models import (
    Distance, VectorParams,
    PointStruct
)
load_dotenv()
from llama_cloud import LlamaCloud

# 2. Membaca Semua File `.md` dari Folder

In [2]:
MD_FOLDER_PATH     = "data_md"                # 📁 Folder berisi file .md
COLLECTION_NAME    = "Thesis Guide for USU Computer Science Students"              # Nama collection Qdrant
EMBEDDING_MODEL    = "paraphrase-multilingual-MiniLM-L12-v2"
VECTOR_DIM         = 384                     # Dimensi vektor MiniLM-L12-v2

# Semantic chunking
WINDOW_SIZE        = 2
MIN_SENTENCES      = 2 
PERCENTILE_THRESH  = 80                      # breakpoint_threshold (percentile)
MAX_CHARS_PER_CHUNK = 1500   

HF_TOKEN = os.getenv("HF_TOKEN")

# Credentials dari .env
QDRANT_URL         = os.getenv("QDRANT_URL")
QDRANT_API_KEY     = os.getenv("QDRANT_API_KEY")

# Validasi
assert QDRANT_URL,     "❌ QDRANT_URL tidak ditemukan di .env!"
assert QDRANT_API_KEY, "❌ QDRANT_API_KEY tidak ditemukan di .env!"


print("✅ Konfigurasi berhasil dimuat!")
print(f"   📁 Folder MD          : {MD_FOLDER_PATH}")
print(f"   ✂️  Max chars/chunk    : {MAX_CHARS_PER_CHUNK}")

✅ Konfigurasi berhasil dimuat!
   📁 Folder MD          : data_md
   ✂️  Max chars/chunk    : 1500


# 3. Parsing Pdf

In [3]:
os.listdir('documents')

['Pedoman tulisan hanya gambar.pdf',
 'Pedoman skripsi tambahan.pdf',
 'Pedoman tulisan tanpa gambar.pdf',
 'Prosedur Tugas Akhir Prodi S-1 Ilmu Komputer.pdf']

In [4]:
client = LlamaCloud()  # Uses LLAMA_CLOUD_API_KEY env var

In [5]:
entries = os.listdir('documents')
files = []
for file in entries:
    with open(f"documents/{file}", "rb") as f:
        doc = f.read()

    result = client.parsing.parse(
        upload_file=doc,
        tier="agentic",
        expand=["markdown_full"], 
        version="latest"
    )
    
    file_name = file.replace('.pdf', '')
    output_file = Path(f"data_md/{file_name}.md")

    # Ensure the 'data_md' directory exists before writing to it
    output_file.parent.mkdir(parents=True, exist_ok=True)

    # Now this will run safely!
    output_file.write_text(result.markdown_full or "")

# 4. Read MD Files

In [6]:
def load_markdown_files(folder_path: str) -> list[dict]:
    """
    Membaca semua file .md dari folder yang ditentukan.
    Returns list of dict: {filename, content}
    """
    md_files = glob.glob(os.path.join(folder_path, "**/*.md"), recursive=True)
    md_files += glob.glob(os.path.join(folder_path, "*.md"))
    md_files = list(set(md_files))  # deduplikasi

    if not md_files:
        raise FileNotFoundError(f"Tidak ada file .md ditemukan di folder: {folder_path}")

    documents = []
    for filepath in sorted(md_files):
        with open(filepath, "r", encoding="utf-8") as f:
            content = f.read().strip()
        if content:
            documents.append({
                "filename": os.path.basename(filepath),
                "filepath": filepath,
                "content": content
            })
            print(f"   📄 {os.path.basename(filepath)} ({len(content):,} karakter)")

    print(f"\n✅ Total {len(documents)} file .md berhasil dibaca!")
    return documents

documents = load_markdown_files(MD_FOLDER_PATH)

   📄 Pedoman skripsi tambahan.md (31,793 karakter)
   📄 Pedoman tulisan hanya gambar.md (9,599 karakter)
   📄 Pedoman tulisan tanpa gambar.md (54,460 karakter)
   📄 Prosedur Tugas Akhir Prodi S-1 Ilmu Komputer.md (10,627 karakter)

✅ Total 4 file .md berhasil dibaca!


# 5. Semantic Chunking

In [7]:
def semantic_chunk_text(
    text: str,
    embed_model: SentenceTransformer,
    window_size: int = 2,
    percentile_threshold: int = 80,
    min_sentences: int = 10
) -> list[str]:
    """
    Melakukan semantic chunking dengan pendekatan percentile breakpoint.

    Algoritma:
    1. Pisahkan teks menjadi kalimat.
    2. Gabungkan kalimat dalam window (sliding window) → group.
    3. Hitung cosine similarity antar group yang berdekatan.
    4. Hitung breakpoint pada persentil ke-N dari distribusi similarity.
    5. Potong teks di titik similarity yang rendah (di bawah threshold).
    6. Gabungkan chunk yang terlalu kecil (< min_sentences) ke chunk berikutnya.
    """
    # Tokenisasi kalimat sederhana
    sentences = [s.strip() for s in re.split(r'(?<=[.!?])\s+', text) if s.strip()]
    if len(sentences) <= window_size:
        return [text]  # Tidak cukup kalimat untuk di-chunk

    # Buat window groups
    groups = []
    for i in range(len(sentences)):
        start = max(0, i - window_size // 2)
        end   = min(len(sentences), i + window_size // 2 + 1)
        groups.append(" ".join(sentences[start:end]))

    # Embedding semua groups
    embeddings = embed_model.encode(groups, show_progress_bar=False, normalize_embeddings=True)

    # Cosine similarity antar group berurutan
    similarities = [
        float(np.dot(embeddings[i], embeddings[i + 1]))
        for i in range(len(embeddings) - 1)
    ]

    # Hitung breakpoint (titik dengan similarity RENDAH = topik berganti)
    distances = [1 - s for s in similarities]
    breakpoint_val = np.percentile(distances, percentile_threshold)

    # Temukan indeks breakpoint
    break_indices = [i + 1 for i, d in enumerate(distances) if d >= breakpoint_val]

    # Potong kalimat berdasarkan breakpoint → hasilkan segmen awal
    segments = []   # list of list[str] (kumpulan kalimat per segmen)
    prev_idx = 0
    for idx in break_indices:
        seg = sentences[prev_idx:idx]
        if seg:
            segments.append(seg)
        prev_idx = idx
    remaining = sentences[prev_idx:]
    if remaining:
        segments.append(remaining)

    if not segments:
        return [text]

    # ── Enforce min_sentences: gabung chunk kecil ke chunk berikutnya ────────
    merged: list[list[str]] = []
    buffer: list[str] = []

    for seg in segments:
        buffer.extend(seg)
        if len(buffer) >= min_sentences:
            merged.append(buffer)
            buffer = []

    # Sisa buffer: jika cukup besar jadikan chunk sendiri,
    # jika tidak gabungkan ke chunk terakhir
    if buffer:
        if merged and len(buffer) < min_sentences:
            merged[-1].extend(buffer)   # gabung ke chunk terakhir
        else:
            merged.append(buffer)       # jadikan chunk baru

    # Konversi list kalimat → string
    chunks = [" ".join(seg).strip() for seg in merged if seg]
    return chunks if chunks else [text]


# ─── Load embedding model ─────────────────────────────────────────────────────
print(f"⏳ Memuat embedding model: {EMBEDDING_MODEL} ...")
embed_model = SentenceTransformer(EMBEDDING_MODEL)
print(f"✅ Model berhasil dimuat! Dimensi vektor: {embed_model.get_sentence_embedding_dimension()}")


# ─── Lakukan semantic chunking untuk semua dokumen ───────────────────────────
all_chunks = []  # List of dict: {chunk_id, text, source_file}

for doc in documents:
    chunks = semantic_chunk_text(
        text=doc["content"],
        embed_model=embed_model,
        window_size=WINDOW_SIZE,
        percentile_threshold=PERCENTILE_THRESH,
        min_sentences=MIN_SENTENCES
    )
    for i, chunk in enumerate(chunks):
        sent_count = len([s for s in re.split(r'(?<=[.!?])\s+', chunk) if s.strip()])
        all_chunks.append({
            "chunk_id"       : f"{doc['filename']}_chunk_{i}",
            "text"           : chunk,
            "source"         : doc["filename"],
            "filepath"       : doc["filepath"],
            "sentence_count" : sent_count
        })
    print(f"   ✂️  {doc['filename']}: {len(chunks)} chunks "
          f"(min {min(len([s for s in re.split(r'(?<=[.!?])\s+', c) if s.strip()]) for c in chunks)} kalimat/chunk)")

print(f"\n✅ Total chunks: {len(all_chunks)}")
if all_chunks:
    counts = [c['sentence_count'] for c in all_chunks]
    print(f"   Rata-rata kalimat/chunk : {sum(counts)/len(counts):.1f}")
    print(f"   Min kalimat/chunk       : {min(counts)}")
    print(f"   Max kalimat/chunk       : {max(counts)}")
print(f"\n📋 Contoh chunk pertama:")
print("-" * 60)
print(all_chunks[0]['text'][:400] if all_chunks else '(kosong)')

⏳ Memuat embedding model: paraphrase-multilingual-MiniLM-L12-v2 ...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

/tmp/ipykernel_16872/1080796167.py:88: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"✅ Model berhasil dimuat! Dimensi vektor: {embed_model.get_sentence_embedding_dimension()}")


✅ Model berhasil dimuat! Dimensi vektor: 384
   ✂️  Pedoman skripsi tambahan.md: 42 chunks (min 2 kalimat/chunk)
   ✂️  Pedoman tulisan hanya gambar.md: 10 chunks (min 2 kalimat/chunk)
   ✂️  Pedoman tulisan tanpa gambar.md: 115 chunks (min 2 kalimat/chunk)
   ✂️  Prosedur Tugas Akhir Prodi S-1 Ilmu Komputer.md: 14 chunks (min 2 kalimat/chunk)

✅ Total chunks: 181
   Rata-rata kalimat/chunk : 5.9
   Min kalimat/chunk       : 2
   Max kalimat/chunk       : 68

📋 Contoh chunk pertama:
------------------------------------------------------------
Abstract blue geometric banner

# PEDOMAN TUGAS AKHIR

PRODI S-1 ILMU KOMPUTER

JUNI
2022

---



<table>
  <thead>
    <tr>
        <th rowspan="5">Universitas Sumatera Utara logo</th>
        <th rowspan="5">PEDOMAN TUGAS<br/>AKHIR/SKRIPSI</th>
        <th>No. Dokumen</th>
        <th>:</th>
        <th> </th>
    </tr>
<tr>
        <th>Edisi</th>
        <th>:</th>
        <th> </th>
    </tr>



# 6. Vector Embedding

In [8]:
print(f"⏳ Melakukan embedding {len(all_chunks)} chunks ...")

texts = [c["text"] for c in all_chunks]

embeddings = embed_model.encode(
    texts,
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True  # Cosine similarity → dot product
)

print(f"\n✅ Embedding selesai!")
print(f"   Shape: {embeddings.shape}")
print(f"   Dimensi: {embeddings.shape[1]}")

⏳ Melakukan embedding 181 chunks ...


Batches:   0%|          | 0/6 [00:00<?, ?it/s]


✅ Embedding selesai!
   Shape: (181, 384)
   Dimensi: 384


# 7. Save to VectorDB (Qdrant)

In [9]:
# ─── Koneksi ke Qdrant Server ─────────────────────────────────────────────────
print(f"⏳ Menghubungkan ke Qdrant di {QDRANT_URL} ...")
qdrant = QdrantClient(
    url=QDRANT_URL,
    api_key=QDRANT_API_KEY,
    timeout=60
)
print("✅ Koneksi berhasil!")


# ─── Buat atau reset collection ───────────────────────────────────────────────
existing_collections = [c.name for c in qdrant.get_collections().collections]

if COLLECTION_NAME in existing_collections:
    print(f"⚠️  Collection '{COLLECTION_NAME}' sudah ada, menghapus dan membuat ulang ...")
    qdrant.delete_collection(COLLECTION_NAME)

qdrant.create_collection(
    collection_name=COLLECTION_NAME,
    vectors_config=VectorParams(
        size=VECTOR_DIM,
        distance=Distance.COSINE
    )
)
print(f"✅ Collection '{COLLECTION_NAME}' berhasil dibuat!")


# ─── Upsert semua vektor ──────────────────────────────────────────────────────
BATCH_SIZE = 100

points = [
    PointStruct(
        id=idx,
        vector=embeddings[idx].tolist(),
        payload={
            "chunk_id" : all_chunks[idx]["chunk_id"],
            "text"     : all_chunks[idx]["text"],
            "source"   : all_chunks[idx]["source"],
            "filepath" : all_chunks[idx]["filepath"]
        }
    )
    for idx in range(len(all_chunks))
]

print(f"\n⏳ Menyimpan {len(points)} vektor ke Qdrant (batch size={BATCH_SIZE}) ...")

for i in tqdm(range(0, len(points), BATCH_SIZE), desc="Upload"):
    batch = points[i:i + BATCH_SIZE]
    qdrant.upsert(collection_name=COLLECTION_NAME, points=batch)

# Verifikasi
count = qdrant.count(collection_name=COLLECTION_NAME).count
print(f"\n✅ Upload selesai! Total vektor tersimpan: {count}")

⏳ Menghubungkan ke Qdrant di https://ea295c4d-3efb-4676-9964-a20d09b4e162.us-east4-0.gcp.cloud.qdrant.io ...
✅ Koneksi berhasil!
⚠️  Collection 'Thesis Guide for USU Computer Science Students' sudah ada, menghapus dan membuat ulang ...
✅ Collection 'Thesis Guide for USU Computer Science Students' berhasil dibuat!

⏳ Menyimpan 181 vektor ke Qdrant (batch size=100) ...


Upload:   0%|          | 0/2 [00:00<?, ?it/s]


✅ Upload selesai! Total vektor tersimpan: 181
